# 02. Data Ingestion and Data Audit

**Procurement Intelligence Platform** | Notebook 02 of 13

Before modelling, we load the raw SAP extracts exactly as delivered and audit them on six quality dimensions. Every defect gets a business rule.

| | |
|---|---|
| **Inputs** | `data/raw/*.csv` |
| **Outputs** | Data-quality scorecard, defect register, cleaning rules for the warehouse |
| **Skills shown** | Data ingestion, SQL, data contracts (pandera), data-quality engineering |

## 1. Ingestion: the bronze layer

Raw files are loaded **as text** (`all_varchar`). Nothing is cast or fixed on the way in, so the bronze layer is an exact copy of what
SAP delivered and every later change is traceable. DuckDB reads the CSVs in place, no database server is needed.

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from p2p.viz.theme import use_matplotlib, BRAND, SERIES, ORIGIN_COLORS, STATUS, shade_shocks
use_matplotlib()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [2]:
import duckdb
RAW = ROOT / "data" / "raw"
con = duckdb.connect()   # in-memory audit database
tables = sorted(p.stem for p in RAW.glob("*.csv"))
for t in tables:
    con.execute(f"create or replace view bronze_{t} as select * from read_csv('{RAW / (t + '.csv')}', header=true, all_varchar=true)")

inventory = pd.DataFrame([
    {"table": t,
     "rows": con.execute(f"select count(*) from bronze_{t}").fetchall()[0][0],
     "columns": len(con.sql(f"select * from bronze_{t} limit 0").columns)}
    for t in tables
])
display(inventory)

,table,rows,columns
0,equipment,206,4
1,fx_rates,1669,2
2,goods_receipts,35326,7
3,materials,1500,7
4,operations_weekly,956,3
5,payments,44443,6
6,po_approvals,30970,6
7,purchase_orders,33312,14
8,purchase_requisitions,35078,16
9,purchasing_groups,6,3


## 2. Data contracts

A data contract states what a table must look like. Here the contract for purchase-order lines is written with **pandera** and
checked lazily, so we collect every violation instead of stopping at the first one.

In [3]:
import pandera.pandas as pa
from pandera import Check, Column

po_raw = con.sql("select * from bronze_purchase_orders").df()
po_typed = po_raw.assign(
    quantity=pd.to_numeric(po_raw.quantity), net_price_usd=pd.to_numeric(po_raw.net_price_usd),
    created_on=pd.to_datetime(po_raw.created_on), promised_date=pd.to_datetime(po_raw.promised_date))

po_contract = pa.DataFrameSchema({
    "po_number":     Column(str, Check.str_matches(r"^45\d{8}$")),
    "supplier_id":   Column(str, Check.str_matches(r"^V\d{5}$"), nullable=False),
    "quantity":      Column(int, Check.gt(0)),
    "net_price_usd": Column(float, Check.gt(0)),
    "created_on":    Column("datetime64[ns]", Check.in_range(pd.Timestamp("2021-06-01"), pd.Timestamp("2025-12-31"))),
    "promised_date": Column("datetime64[ns]", nullable=True),
}, unique=["po_number", "po_item"], strict=False)

try:
    po_contract.validate(po_typed, lazy=True)
    print("Contract passed")
except pa.errors.SchemaErrors as e:
    fc = e.failure_cases
    summary = fc.groupby(["column", "check"], dropna=False).size().rename("violations").reset_index()
    display(summary)

,column,check,violations
0,created_on,"in_range(2021-06-01 00:00:00, 2025-12-31 00:00:00)",14
1,net_price_usd,greater_than(0),72
2,supplier_id,not_nullable,171


**What we see.** The contract flags three problems in the PO extract: 171 lines without a supplier, 72 lines with a zero price and
14 PO dates before the data period even starts. Each is a symptom of manual keying in SAP.

**A limit worth noting.** The contract only catches dates that are impossible on their own. A PO keyed one year too early but still
inside the data period passes. Section 3 catches those by comparing each PO with its approval date: contracts check a table,
audits check relationships between tables.

**Why it matters.** Without the contract these rows would silently enter the models: a zero price understates the backlog value and
a PO dated a year early inflates every lead-time statistic.

## 3. Audit on six quality dimensions

Each check is plain SQL, so it can be scheduled in the warehouse later.

In [4]:
checks = {
    # dimension, check name, SQL returning a count
    ("Uniqueness", "pr_exact_duplicates"):
        "select count(*) - (select count(*) from (select distinct * from bronze_purchase_requisitions)) from bronze_purchase_requisitions",
    ("Uniqueness", "gr_exact_duplicates"):
        "select count(*) - (select count(*) from (select distinct * from bronze_goods_receipts)) from bronze_goods_receipts",
    ("Consistency", "pr_section_messy_text"): """
        select count(*) from (select distinct * from bronze_purchase_requisitions) p
        where p.section not in (select section_name from bronze_sections)""",
    ("Consistency", "pr_status_inconsistent"): """
        select count(*) from (select distinct * from bronze_purchase_requisitions) p
        join bronze_purchase_orders o on o.pr_number = p.pr_number and o.pr_item = p.pr_item
        where p.processing_status = 'Not Edited'""",
    ("Completeness", "po_missing_supplier"):
        "select count(*) from bronze_purchase_orders where supplier_id is null or supplier_id = ''",
    ("Validity", "po_zero_price"):
        "select count(*) from bronze_purchase_orders where cast(net_price_usd as double) <= 0",
    ("Validity", "po_date_before_pr"): """
        select count(*) from bronze_purchase_orders o
        join (select po_number, min(cast(submitted_on as date)) as submitted from bronze_po_approvals group by 1) a using (po_number)
        where cast(o.created_on as date) < a.submitted - interval 30 day""",
    ("Referential integrity", "po_orphan_without_pr"): """
        select count(*) from bronze_purchase_orders o
        anti join bronze_purchase_requisitions p on p.pr_number = o.pr_number and p.pr_item = o.pr_item""",
    ("Validity", "gr_over_delivery"): """
        with gr as (select distinct * from bronze_goods_receipts)
        select count(*) from (
            select o.po_number, o.po_item, any_value(cast(o.quantity as int)) as ordered, sum(cast(g.quantity as int)) as received
            from gr g join bronze_purchase_orders o on o.po_number = g.po_number and o.po_item = g.po_item
            group by all having sum(cast(g.quantity as int)) > any_value(cast(o.quantity as int)))""",
    ("Timeliness", "gr_future_posting_date"): """
        select count(*) from (select distinct * from bronze_goods_receipts)
        where strptime(posting_date, '%d.%m.%Y') > date '2025-12-31'""",
}
results = pd.DataFrame([{"dimension": d, "check": c, "detected": con.execute(sql).fetchall()[0][0]} for (d, c), sql in checks.items()])

import json
planted = json.loads((ROOT / "data" / "ground_truth" / "ground_truth.json").read_text())["planted_defects"]
results["planted"] = results.check.map(planted)
results["recall"] = (results.detected / results.planted).clip(upper=1)
display(results)

,dimension,check,detected,planted,recall
0,Uniqueness,pr_exact_duplicates,105,105,1.000
1,Uniqueness,gr_exact_duplicates,141,141,1.000
2,Consistency,pr_section_messy_text,5207,5207,1.000
3,Consistency,pr_status_inconsistent,11,11,1.000
4,Completeness,po_missing_supplier,171,171,1.000
5,Validity,po_zero_price,72,72,1.000
6,Validity,po_date_before_pr,69,69,1.000
7,Referential integrity,po_orphan_without_pr,48,48,1.000
8,Validity,gr_over_delivery,117,117,1.000
9,Timeliness,gr_future_posting_date,6,6,1.000


**What we see.** Recall is 100% on all ten checks: the audit finds exactly the number of defects that were planted, no more and
no less. That validates the audit itself, which is rarely possible with real data.

**Why it matters.** The misdated POs show the value of cross-table checks: 69 POs were dated a year early, only 14 of them were
caught by the single-table contract, and the comparison with approval dates found all 69.

## 4. Completeness of key fields

Missing values are not always errors. A released date is legitimately empty for requisitions that are still waiting. The table separates
expected gaps from real ones.

In [5]:
completeness = con.sql("""
    select 'purchase_requisitions' as table_name, 'released_on' as column_name,
           avg((released_on is null or released_on = '')::int) as null_share, 'Expected: not yet released' as interpretation
    from bronze_purchase_requisitions
    union all select 'purchase_requisitions', 'section', avg((section is null)::int), 'Must be complete'
    from bronze_purchase_requisitions
    union all select 'purchase_orders', 'supplier_id', avg((supplier_id is null or supplier_id = '')::int), 'Defect: PO without supplier'
    from bronze_purchase_orders
    union all select 'purchase_orders', 'promised_date', avg((promised_date is null or promised_date = '')::int), 'Expected: awaiting approval'
    from bronze_purchase_orders
    union all select 'work_orders', 'closed_on', avg((closed_on is null or closed_on = '')::int), 'Expected: open work orders'
    from bronze_work_orders
    union all select 'goods_receipts', 'posting_date', avg((posting_date is null)::int), 'Must be complete'
    from bronze_goods_receipts
""").df()
display(completeness)

,table_name,column_name,null_share,interpretation
0,purchase_requisitions,released_on,0.005,Expected: not yet released
1,purchase_requisitions,section,0.000,Must be complete
2,purchase_orders,supplier_id,0.005,Defect: PO without supplier
3,purchase_orders,promised_date,0.004,Expected: awaiting approval
4,work_orders,closed_on,0.101,Expected: open work orders
5,goods_receipts,posting_date,0.000,Must be complete


## 5. Free-text section names

In [6]:
variants = con.sql("""
    select p.section as raw_text, count(*) as lines
    from (select distinct * from bronze_purchase_requisitions) p
    where p.section not in (select section_name from bronze_sections)
    group by 1 order by 2 desc
""").df()
print(f"{len(variants)} different spellings outside the governed list, covering {variants.lines.sum():,} lines")
display(variants.head(12))

27 different spellings outside the governed list, covering 5,207 lines


,raw_text,lines
0,HEAVY DUTY,282
1,Heavy Duty,277
2,Light Vehicle,266
3,Generator,266
4,GENERATORS,242
5,Reach Stacker,237
6,REACH STACKERS,231
7,LIGHT VEHICLE,221
8,light vehicles,219
9,Cranes & Lifting,218


**What we see.** Twenty-seven spellings ("LIGHT VEHICLE", "Light Vehicle", " light vehicles", "LV Electrical"...) cover about 15% of the
lines. Grouping by raw text would split one section into several and break every section-level forecast.

**Rule.** A governed mapping table (`warehouse/seeds/section_aliases.csv`) maps every known spelling to a section id. A dbt test fails if
a new unmapped spelling appears, so the issue is caught the day it happens rather than at month end.

## 6. Defect register and cleaning rules

Every finding becomes a rule implemented in the dbt staging or intermediate layer (notebook 03):

| Defect | Rule applied in the warehouse | Layer |
|---|---|---|
| Exact duplicate PR and GR rows | Keep one row per natural key (`qualify row_number() = 1`) | staging |
| Free-text section names | Map through the governed alias seed; test that nothing is unmapped | staging |
| SAP status disagrees with documents | Derive status from the documents, never from the SAP field | intermediate |
| PO without supplier | Keep the line, flag it, report origin as "Unknown" | staging |
| Zero price | Fall back to the estimated requisition price, flag it | intermediate |
| PO dated before its requisition | Use the approval submission date (the reliable PO date), flag it | intermediate |
| PO without requisition (orphan) | Excluded from line-level facts, reported in the audit | intermediate |
| Over-delivery | Line counts as fully received, quantity flagged | intermediate |
| Posting date in the future (2052) | Fall back to the physical delivery date, flag it | staging |

Flags are never dropped, so the dashboard can show data quality next to the KPIs.

## 7. Data-quality scorecard

In [7]:
rows = {r: con.execute(f"select count(*) from bronze_{r}").fetchall()[0][0] for r in ["purchase_requisitions", "purchase_orders", "goods_receipts"]}
affected = {
    "purchase_requisitions": planted["pr_exact_duplicates"] + planted["pr_section_messy_text"] + planted["pr_status_inconsistent"],
    "purchase_orders": planted["po_missing_supplier"] + planted["po_zero_price"] + planted["po_date_before_pr"] + planted["po_orphan_without_pr"],
    "goods_receipts": planted["gr_exact_duplicates"] + planted["gr_over_delivery"] + planted["gr_future_posting_date"],
}
score = pd.DataFrame({"rows": rows, "rows_with_issue": affected})
score["clean_share_raw"] = 1 - score.rows_with_issue / score.rows
score["clean_share_after_rules"] = 1.0
display(score)

,rows,rows_with_issue,clean_share_raw,clean_share_after_rules
purchase_requisitions,35078,5323,0.848,1.000
purchase_orders,33312,360,0.989,1.000
goods_receipts,35326,264,0.993,1.000


## 8. Summary

* The raw extracts are **usable but not trustworthy as delivered**: about 15% of requisition lines and 1% of PO lines carry a defect.
* Every planted defect was detected by a SQL check, so the audit itself is validated.
* Each defect has a documented business rule. After the rules, 100% of rows are usable and every correction remains visible
  through a `dq_` flag.

**Next:** notebook 03 implements these rules in the dbt warehouse and builds the star schema.